# Práctica 2 — Reconocimiento Automático del Habla

**Procesamiento del Habla · Grado en Ingeniería Informática · ETSI · Universidad de Huelva**

Este cuaderno es el punto de partida. El enunciado completo, con las tareas y la
rúbrica, está en el documento de la práctica publicado en Moodle.

---

### Antes de nada: las tres cosas que ahorran una sesión

1. **`float16`, nunca `int8`.** La GPU de NIMROD es Blackwell (sm_120) y
   faster-whisper con `compute_type="int8"` falla ahí.
2. **No forcéis descargas de modelos.** Los pesos están pre-descargados en la
   caché compartida, que es de sólo lectura. Ocho equipos bajando `large-v3` a la
   vez saturan la red.
3. **Medid el rendimiento con el protocolo**, no a ojo: carga aparte, pasadas de
   calentamiento descartadas y VRAM por NVML. Los tres detalles están explicados
   donde tocan.

### Estructura

| Sesión | Qué se hace |
|---|---|
| 1 | Baseline y normalización |
| 2 | Comparativa de tres modelos |
| 3 | Robustez y cierre |


In [1]:
# Celda de arranque. Ejecutadla SIEMPRE, antes de cargar ningún modelo.
import os, sys, pathlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from scripts.entorno import verificar_entorno
from scripts.asr_util import (cargar_manifiesto, dispositivo, duracion_total_s,
                              sin_ruido, silenciar_avisos_cache)

# La caché de modelos de NIMROD es de sólo lectura a propósito. Hugging Face
# intenta escribir metadatos en ella, no puede, lo ignora y continúa — pero
# avisa cada vez y llena la salida. Esto quita ESOS avisos y nada más.
silenciar_avisos_cache()
from scripts.metrics import (Resultado, MedidorVRAM, cronometro, calentar,
                             evaluar, detectar_entorno, guardar_csv)
from scripts.normalize import normalizar

verificar_entorno()

# El conjunto se lee de P2_EVAL_SET, que en NIMROD apunta al conjunto comun
# completo. En local esa variable no existe: si nos hemos traido el conjunto
# de NIMROD a eval_set/, se usa ese, y si no, quedan las 12 muestras de
# eval_local/. Asi el cuaderno funciona en los dos sitios sin configurar
# variables de entorno en el IDE, que es donde mas se falla.
if "P2_EVAL_SET" not in os.environ and Path("eval_set/manifest.tsv").exists():
    os.environ["P2_EVAL_SET"] = "eval_set"

EVAL_SET = os.environ.get("P2_EVAL_SET", "eval_local")

# El dispositivo NO se escribe a mano: en NIMROD hay que usar float16 (la GPU
# es Blackwell y falla con int8), y en un portátil sin GPU float16 no existe.
DEVICE, PRECISION = dispositivo()

print(f"\nconjunto de evaluación: {EVAL_SET}")
if EVAL_SET == "eval_local":
    print("  [AVISO] son las 12 muestras de prueba. Valen para depurar, NO para")
    print("          las cifras de la entrega, que salen de los 200 enunciados.")
print(f"dispositivo: {DEVICE} / {PRECISION}")


  OK     entorno      local — WER y CER válidos; RTF y VRAM hay que medirlos en NIMROD
  OK     python       3.12.10
  OK     paquetes     todos los requeridos presentes
  OK     backend      faster-whisper disponible
  OK     ffmpeg       C:\ffmpeg\ffmpeg.EXE
  OK     gpu          NVIDIA GeForce RTX 3070 Laptop GPU — 8.0 GB, sm_86
  OK     dispositivo  cuda / compute_type="float16"
  OK     cache_hf     (por defecto, local)
  OK     eval_set     12 enunciados, 1.9 min, bloques: fleurs, voxpopuli
  OK     límites      sin límite de cgroup

Entorno listo.
Recordad: RTF, tiempo de carga y VRAM se miden en NIMROD.

conjunto de evaluación: eval_local
  [AVISO] son las 12 muestras de prueba. Valen para depurar, NO para
          las cifras de la entrega, que salen de los 200 enunciados.
dispositivo: cuda / float16


In [2]:
EQUIPO = "equipo-02"   # <- RELLENAD con el nombre de vuestro equipo


---
# Sesión 1 — Baseline y normalización

**Objetivo:** dejar el pipeline de medida funcionando de punta a punta y obtener
una cifra de WER de referencia con un modelo pequeño.


## 1.1 El normalizador

`scripts/normalize.py` es la **única** fuente de verdad y **no se modifica**.
Se aplica por igual a la hipótesis y a la referencia.

Entended qué hace antes de seguir: si no, los WER que midáis no significarán lo
que creéis.

In [3]:
for t in ["El tren de las 23:15 llegó con 3,5 minutos de retraso.",
          "¿Cuánto cuesta? 1.200 € —casi un 15 % más—.",
          "La señora Núñez vive en Aljaraque."]:
    print(f"{t}\n  -> {normalizar(t)}\n")


El tren de las 23:15 llegó con 3,5 minutos de retraso.
  -> el tren de las veintitrés quince llegó con tres coma cinco minutos de retraso

¿Cuánto cuesta? 1.200 € —casi un 15 % más—.
  -> cuánto cuesta mil doscientos euros casi un quince por ciento más

La señora Núñez vive en Aljaraque.
  -> la señora núñez vive en aljaraque



### Por qué importa

Dos modelos pueden acertar exactamente lo mismo y dar WER muy distintos si uno
emite cifras y el otro palabras. Compruébalo:

In [4]:
ref = ["Llegaron veintitrés personas"]
print("hipótesis en cifras:", evaluar(ref, ["Llegaron 23 personas"]))
print("hipótesis en letras:", evaluar(ref, ["Llegaron veintitrés personas"]))
# Las dos son correctas. Tras normalizar, el WER es 0 en ambas.


hipótesis en cifras: (0.0, 0.0)
hipótesis en letras: (0.0, 0.0)


## 1.2 El conjunto de evaluación

In [5]:
enunciados = cargar_manifiesto(f"{EVAL_SET}/manifest.tsv")
dur_total = duracion_total_s(enunciados)
refs = [e.transcripcion_referencia for e in enunciados]

print(f"{len(enunciados)} enunciados, {dur_total/60:.1f} minutos de audio")

from collections import Counter
for bloque, n in Counter(e.bloque for e in enunciados).items():
    print(f"  {bloque}: {n}")


12 enunciados, 1.9 minutos de audio
  fleurs: 6
  voxpopuli: 6


## 1.3 Baseline

Modelo pequeño, condición `limpio`. Aquí no interesa el rendimiento todavía:
interesa que el pipeline entero funcione.

In [6]:
from faster_whisper import WhisperModel

with sin_ruido():
    modelo_base = WhisperModel("small", device=DEVICE, compute_type=PRECISION)

def transcribir_base(ruta):
    with sin_ruido():
        segs, _ = modelo_base.transcribe(str(ruta), language="es", beam_size=5)
        return " ".join(s.text for s in segs).strip()

print("modelo cargado")


modelo cargado


In [7]:
hipotesis = []
for i, e in enumerate(enunciados):
    hipotesis.append(transcribir_base(Path(EVAL_SET) / e.ruta))
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(enunciados)}")

wer, cer = evaluar(refs, hipotesis)
print(f"\nWER baseline: {wer:.4f}")
print(f"CER baseline: {cer:.4f}")



WER baseline: 0.0515
CER baseline: 0.0211


## 1.4 Mirad los errores, no sólo la cifra

Un WER agregado no dice dónde falla el modelo. Ordenad por error y leed los diez
peores: casi siempre hay un patrón (números, nombres propios, un bloque entero
del corpus, alucinación al final del audio).

In [8]:
import jiwer

peores = sorted(
    ((jiwer.wer(normalizar(e.transcripcion_referencia), normalizar(h)), e, h)
     for e, h in zip(enunciados, hipotesis)
     if normalizar(e.transcripcion_referencia).strip()),
    key=lambda x: -x[0])[:10]

for w, e, h in peores:
    print(f"[{w:.2f}] {e.bloque}")
    print(f"  ref: {e.transcripcion_referencia}")
    print(f"  hip: {h}\n")


[0.23] voxpopuli
  ref: Señora Presidenta, ante todo, gracias a la ponente. Dicho lo cual, resulta increíble que, siendo ponente alternativo de este informe, no tenga tiempo asignado para el debate en el Pleno.
  hip: Ante todo, gracias al aponente, dicho lo cual, resulta increíble que, siendo oponente alternativo de este informe, no tenga tiempo asignado para el debate en pleno.

[0.07] fleurs
  ref: en el caso de los springboks significó superar una seguidilla de cinco encuentros consecutivos perdidos
  hip: En el caso de los Springbox, significó superar una seguidilla de 5 encuentros consecutivos perdidos.

[0.06] fleurs
  ref: los pensamientos de aristóteles en especial las cuestiones de la ciencia incluso la psicología fueron aceptadas
  hip: Los pensamientos de Aristóteles, en especial las cuestiones de la ciencia e incluso la  psicología, fueron aceptadas.

[0.06] fleurs
  ref: a menor número de rossby menor es la actividad de la estrella en relación con sus inversiones magnétic

In [11]:
# Liberamos el baseline antes de seguir: la GPU es compartida.
del modelo_base, transcribir_base
import gc
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
except ImportError:
    pass   # sin torch no hay nada que liberar; faster-whisper no lo necesita
print("memoria liberada")


NameError: name 'modelo_base' is not defined

---
# Sesión 2 — Comparativa de tres modelos

Tres modelos de **familias arquitectónicas distintas**, condición `limpio`, más
la pasada de rendimiento.

| Métrica | Dónde se mide |
|---|---|
| WER, CER | donde queráis |
| RTF, t_carga, VRAM pico | **obligatoriamente en NIMROD** |

No es control: un RTF de 0,30 en un portátil sin GPU y uno de 0,05 en una tarjeta
de gama media no dicen nada el uno del otro.

## 2.1 Los backends

Aquí están implementados **dos** familias, que son las que vienen pre-descargadas
en la caché de NIMROD:

- **encoder-decoder** vía faster-whisper (CTranslate2);
- **CTC** vía transformers (wav2vec2 / MMS).

Son el ejemplo del que partir. **El tercer modelo lo añadís vosotros**, y debe ser
de una familia distinta a las dos anteriores: ampliad `cargar()` y `transcribir()`
con el backend que necesite.

### Nuestro tercer modelo: Vosk (Kaldi, híbrido TDNN-HMM)

Las dos familias que trae la plantilla son **encoder-decoder** (Whisper) y **CTC
end-to-end** (wav2vec2). Elegimos como tercera un modelo **híbrido clásico**:
modelo acústico TDNN + diccionario de pronunciación + modelo de lenguaje n-grama
compilado en un WFST. No es end-to-end, y esa es exactamente la diferencia que
buscábamos.

Nota sobre `facebook/mms-1b-all`: el comentario de la plantilla lo propone como
tercer candidato, pero MMS es **también CTC**, la misma familia que wav2vec2. No
lo usamos como tercer modelo por eso.

Dos consecuencias de elegir un modelo de CPU que hay que tener presentes:

- **No reserva VRAM.** Lo que `MedidorVRAM` mida en su fila será la memoria que
  el proceso siga teniendo reservada por el contexto CUDA de los modelos
  anteriores, no la de Vosk. Queda anotado en la ficha y en el informe.
- **El RTF sigue midiéndose en NIMROD**, aunque corra en CPU: lo comparable es
  el mismo hardware para todos, y el contenedor de NIMROD tiene su propio
  límite de núcleos (`cpus_limite()`).


In [ ]:
# --- Modelo Vosk: localizarlo o descargarlo --------------------------------
# Vosk NO está en la caché compartida de NIMROD, así que el modelo lo traemos
# nosotros. Va a modelos/, que el .gitignore ya excluye del repositorio.
import json as _json
import urllib.request
import zipfile

# Grande (1,4 GB). Si vais justos de disco o de tiempo, cambiad a
# "vosk-model-small-es-0.42", que son 39 MB y bastante peor: anotadlo entonces
# en la ficha, porque no es el mismo modelo.
MODELO_VOSK = "vosk-model-es-0.42"
DIR_MODELOS = Path("modelos")
URL_VOSK = "https://alphacephei.com/vosk/models"


def ruta_modelo_vosk(nombre):
    """Devuelve la carpeta del modelo Vosk. No descarga nada."""
    candidatas = [DIR_MODELOS / nombre, Path(nombre), Path.home() / "modelos" / nombre]
    desde_entorno = os.environ.get("VOSK_MODEL_PATH")
    if desde_entorno:
        candidatas.insert(0, Path(desde_entorno))

    for i in range(len(candidatas)):
        if candidatas[i].is_dir():
            return candidatas[i]

    raise FileNotFoundError(
        f"No encuentro el modelo Vosk '{nombre}'. Ejecutad la celda de descarga, "
        f"o descomprimidlo a mano en {DIR_MODELOS / nombre}"
    )


def descargar_modelo_vosk(nombre):
    """Descarga y descomprime el modelo si no está ya. Idempotente."""
    destino = DIR_MODELOS / nombre
    if destino.is_dir():
        print(f"ya está: {destino}")
        return destino

    DIR_MODELOS.mkdir(parents=True, exist_ok=True)
    zip_local = DIR_MODELOS / f"{nombre}.zip"
    if not zip_local.exists():
        print(f"descargando {nombre} ...")
        urllib.request.urlretrieve(f"{URL_VOSK}/{nombre}.zip", zip_local)

    print("descomprimiendo ...")
    with zipfile.ZipFile(zip_local) as z:
        z.extractall(DIR_MODELOS)
    zip_local.unlink(missing_ok=True)
    print(f"listo: {destino}")
    return destino


descargar_modelo_vosk(MODELO_VOSK)


In [ ]:
import torch
import soundfile as sf
import wave


def cargar(nombre, backend, precision):
    """Devuelve un objeto de modelo listo para transcribir."""
    if backend == "faster-whisper":
        from faster_whisper import WhisperModel
        with sin_ruido():
            return WhisperModel(nombre, device=DEVICE, compute_type=precision)

    if backend == "transformers":
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        with sin_ruido():
            proc = AutoProcessor.from_pretrained(nombre)
            modelo = Wav2Vec2ForCTC.from_pretrained(
                nombre,
                dtype=torch.float16 if precision == "float16" else torch.float32,
            ).to(DEVICE).eval()
            # MMS publica un adaptador por idioma; sin cargarlo transcribe en inglés.
            if "mms" in nombre.lower():
                proc.tokenizer.set_target_lang("spa")
                modelo.load_adapter("spa")
        return (proc, modelo)

    # Vosk (Kaldi): familia híbrida TDNN-HMM. Es CPU, así que el argumento
    # `precision` no se usa: lo dejamos en "int8" en la fila del CSV para
    # reflejar que los pesos del modelo acústico ya vienen cuantizados.
    if backend == "vosk":
        from vosk import Model as ModeloVosk
        from vosk import SetLogLevel

        # Vosk escribe su log desde C++ por stderr; sin_ruido() no basta porque
        # parte del volcado ocurre al construir el modelo.
        SetLogLevel(-1)
        with sin_ruido():
            return ModeloVosk(str(ruta_modelo_vosk(nombre)))

    raise NotImplementedError(
        f"Backend '{backend}' no implementado. Añadidlo aquí si lo necesitáis."
    )


def transcribir(modelo, backend, ruta):
    """Devuelve el texto crudo, SIN normalizar."""
    if backend == "faster-whisper":
        with sin_ruido():
            segs, _ = modelo.transcribe(str(ruta), language="es", beam_size=5)
            return " ".join(s.text for s in segs).strip()

    if backend == "transformers":
        proc, red = modelo
        audio, sr = sf.read(str(ruta), dtype="float32")
        entradas = proc(audio, sampling_rate=sr, return_tensors="pt")
        entradas = {k: v.to(DEVICE) for k, v in entradas.items()}
        if red.dtype == torch.float16 and "input_values" in entradas:
            entradas["input_values"] = entradas["input_values"].half()
        with torch.no_grad():
            logits = red(**entradas).logits
        ids = torch.argmax(logits, dim=-1)
        return proc.batch_decode(ids)[0].strip()

    if backend == "vosk":
        from vosk import KaldiRecognizer

        # El reconocedor es un objeto con estado: uno nuevo por enunciado. El
        # modelo, que es lo caro de cargar, se reutiliza.
        with wave.open(str(ruta), "rb") as onda:
            reconocedor = KaldiRecognizer(modelo, onda.getframerate())
            reconocedor.SetWords(False)
            while True:
                trozo = onda.readframes(4000)
                if len(trozo) == 0:
                    break
                reconocedor.AcceptWaveform(trozo)
            final = _json.loads(reconocedor.FinalResult())
        return final.get("text", "").strip()

    raise NotImplementedError(f"Backend '{backend}' no implementado.")


In [ ]:
# Comprobación rápida de los TRES backends sobre UN enunciado, antes de lanzar
# la pasada completa. Si algo falla, que falle aquí y no a los 15 minutos.
_prueba = Path(EVAL_SET) / enunciados[0].ruta
print("ref:", enunciados[0].transcripcion_referencia, "\n")

for _n, _b, _p in [("small", "faster-whisper", PRECISION),
                   ("facebook/wav2vec2-large-xlsr-53-spanish", "transformers", "float32"),
                   (MODELO_VOSK, "vosk", "int8")]:
    try:
        _m = cargar(_n, _b, _p)
        print(f"{_b:<16} {transcribir(_m, _b, _prueba)}")
        del _m
        gc.collect(); torch.cuda.empty_cache()
    except Exception as ex:
        print(f"{_b:<16} FALLO: {type(ex).__name__}: {ex}")


## 2.2 Vuestros tres modelos

Las dos primeras líneas funcionan tal cual. **Sustituid o añadid la tercera**: no
vale entregar dos modelos, ni tres de la misma familia.

Comprobad vosotros la licencia de cada uno. Es uno de los seis criterios y alguna
de las de abajo puede haber cambiado.

In [ ]:
MODELOS = [
    # (nombre, backend, precision, licencia)
    ("large-v3-turbo",                          "faster-whisper", PRECISION, "MIT"),
    ("facebook/wav2vec2-large-xlsr-53-spanish", "transformers",   "float32", "Apache-2.0"),

    # Tercera familia: híbrido TDNN-HMM (Kaldi) vía Vosk. No es encoder-decoder
    # ni CTC end-to-end. Corre en CPU, de ahí "int8": los pesos del modelo
    # acústico ya vienen cuantizados en el paquete.
    (MODELO_VOSK,                               "vosk",           "int8",    "Apache-2.0"),
]

assert len({m[0] for m in MODELOS}) >= 3, "Hacen falta tres modelos"
print(f"{len(MODELOS)} modelos:")
for n, b, p, lic in MODELOS:
    print(f"  {n}  ({b}, {p}, {lic})")


## 2.3 Pasada completa

Tres cosas que esta celda hace a propósito:

- **`t_carga` se mide aparte** de la inferencia: es un coste de una sola vez y
  mezclarlo destroza el RTF.
- **`calentar()` descarta dos pasadas** antes de cronometrar. La primera
  inferencia paga el contexto CUDA y la compilación de kernels: medido en NIMROD,
  RTF 1,403 en la primera pasada frente a 0,053 en las siguientes. Un factor de 26
  que no tiene nada que ver con el modelo.
- **La VRAM se muestrea con NVML**, no con `torch.cuda.max_memory_allocated()`,
  que no ve a CTranslate2 y devolvería 0 MB sin avisar.

In [ ]:
entorno = detectar_entorno()
mide_rendimiento = entorno == "nimrod"
resultados = []

for nombre, backend, precision, licencia in MODELOS:
    print(f"\n=== {nombre} ({backend}, {precision}) ===")

    with cronometro() as t_carga:
        modelo_i = cargar(nombre, backend, precision)
    print(f"  carga: {t_carga.segundos:.1f} s")

    # Pasadas de calentamiento DESCARTADAS.
    calentar(lambda r: transcribir(modelo_i, backend, r),
             Path(EVAL_SET) / enunciados[0].ruta)

    with MedidorVRAM() as vram, cronometro() as t_inf:
        hips = [transcribir(modelo_i, backend, Path(EVAL_SET) / e.ruta)
                for e in enunciados]

    wer, cer = evaluar(refs, hips)
    rtf = t_inf.segundos / dur_total
    print(f"  WER {wer:.4f} | CER {cer:.4f} | RTF {rtf:.4f} | VRAM {vram.pico_mb} MB")

    resultados.append(Resultado(
        equipo=EQUIPO, modelo=nombre, backend=backend, precision=precision,
        condicion="limpio", entorno=entorno, wer=wer, cer=cer,
        # Fuera de NIMROD estas tres columnas van VACÍAS. No las estiméis.
        rtf_medio=rtf if mide_rendimiento else None,
        t_carga_s=t_carga.segundos if mide_rendimiento else None,
        vram_pico_mb=vram.pico_mb if mide_rendimiento else None,
        licencia=licencia,
    ))

    del modelo_i
    gc.collect(); torch.cuda.empty_cache()


In [ ]:
guardar_csv(resultados, "resultados/resultados.csv")

import pandas as pd
pd.DataFrame([r.fila() for r in resultados])


**Mirad los errores del modelo CTC antes de sacar conclusiones.** En una
prueba sobre este mismo conjunto, el CTC transcribió «sicología» donde la
referencia dice «psicología». Las dos grafías son correctas en español, así que
eso cuenta como error de WER **sin ser un error de reconocimiento**.

Es un buen recordatorio de lo que mide el WER: distancia a una referencia
concreta, no corrección. ¿Debería el normalizador unificar variantes
ortográficas? Pensadlo y dejadlo argumentado en la ficha.

**Y un aviso que cuesta caro olvidar:** si medís el WER en vuestra máquina con un
backend y el RTF en NIMROD con otro, estáis comparando dos implementaciones
distintas del mismo modelo. `comprobar_entrega.py` os lo rechaza.

---
# Sesión 3 — Robustez y cierre

Las seis condiciones acústicas, el round-trip opcional con P1 y el cierre del
entregable.

## 3.1 Generar las degradaciones

Una sola vez. Es determinista: la semilla está fijada, así que el audio degradado
sale idéntico en cualquier máquina.

In [ ]:
import subprocess

if not Path("eval_degradado/snr10").exists():
    r = subprocess.run(["python3", "-I", "scripts/degradar.py",
                        "--eval-set", EVAL_SET, "--salida", "eval_degradado"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print("ya generado")


## 3.2 Escuchad el audio degradado

Antes de medir, escuchad. Si a 5 dB de SNR vosotros no entendéis la frase, no
esperéis que el modelo la acierte, y eso ya es parte del análisis.

In [ ]:
from IPython.display import Audio, display

e0 = enunciados[0]
print(e0.transcripcion_referencia)
for cond in ["limpio", "snr10", "snr5", "telefono", "codec"]:
    base = f"{EVAL_SET}/audio" if cond == "limpio" else f"eval_degradado/{cond}"
    print(cond)
    display(Audio(f"{base}/{Path(e0.ruta).name}"))


## 3.3 Pasada sobre las seis condiciones

In [ ]:
CONDICIONES = ["limpio", "snr20", "snr10", "snr5", "telefono", "codec"]

def ruta_de(e, cond):
    if cond == "limpio":
        return Path(EVAL_SET) / e.ruta
    return Path("eval_degradado") / cond / Path(e.ruta).name

filas = list(resultados)   # conserva las filas de rendimiento de la sesión 2

for nombre, backend, precision, licencia in MODELOS:
    modelo_i = cargar(nombre, backend, precision)
    for cond in CONDICIONES:
        if cond == "limpio":
            continue   # ya medida en la sesión 2
        hips = [transcribir(modelo_i, backend, ruta_de(e, cond)) for e in enunciados]
        wer, cer = evaluar(refs, hips)
        print(f"{nombre:<42} {cond:<10} WER {wer:.4f}")
        filas.append(Resultado(
            equipo=EQUIPO, modelo=nombre, backend=backend, precision=precision,
            condicion=cond, entorno=entorno, wer=wer, cer=cer, licencia=licencia))
    del modelo_i
    gc.collect(); torch.cuda.empty_cache()

guardar_csv(filas, "resultados/resultados.csv")


## 3.4 La curva de degradación

Es el gráfico que de verdad responde a «qué modelo aguanta mejor». Un modelo con
WER bajo en `limpio` que se desploma a 5 dB puede ser peor elección que uno
mediocre pero estable.

In [ ]:
import pandas as pd

df = pd.DataFrame([f.fila() for f in filas])
tabla = df.pivot_table(index="condicion", columns="modelo", values="wer")
tabla = tabla.reindex(CONDICIONES)
display(tabla)

tabla.plot(marker="o", figsize=(9, 5),
           title="WER por condición acústica", ylabel="WER", xlabel="")


## 3.5 Round-trip con P1 (opcional, bonificable)

Sintetizad las referencias del conjunto común con las voces Piper y MMS-TTS de
P1 y transcribidlas. Mide el ciclo completo texto → TTS → ASR → texto.

La pregunta interesante no es el WER en sí, sino **dónde se pierde**: si un error
aparece sólo con una voz concreta, el fallo es del sintetizador; si aparece con
todas, es del reconocedor.

In [ ]:
# Reutilizad vuestro código de P1 para generar el audio en roundtrip/<voz>/ y:
#   hips = [transcribir(modelo_i, backend, f"roundtrip/{voz}/{e.id}.wav")
#           for e in enunciados]
#   wer, cer = evaluar(refs, hips)


## 3.6 Cierre

Rellenad `informe/ficha_modelos.md`, dejad `transcribe.py` funcionando con
vuestro modelo elegido y comprobad la entrega:

```bash
python3 comprobar_entrega.py
```

- [ ] `resultados/resultados.csv` completo (≥ 18 filas, esquema fijo)
- [ ] Al menos una fila `entorno=nimrod`, `condicion=limpio` por modelo, con RTF,
      t_carga y VRAM rellenos
- [ ] `informe/ficha_modelos.md` con los seis criterios por modelo
- [ ] `transcribe.py` funcionando, con carga cacheada
- [ ] `comprobar_entrega.py` sin problemas